The objective of this notebook is to create meaningful business features from the cleaned Flipkart datasets. These engineered features enhance customer, sales, product, and logistics analysis while preparing the data for SQL queries, advanced analytics, and Power BI dashboards.

In [1]:
## import libraries
import pandas as pd
import numpy as np

In [2]:
## load cleaned dataset
customers = pd.read_csv("D:\Flipkart sales analytics\cleaned_dataset\Customers.csv")
orders = pd.read_csv("D:\Flipkart sales analytics\cleaned_dataset\Orders.csv")
order_items = pd.read_csv("D:\Flipkart sales analytics\cleaned_dataset\OrderItems.csv")
products = pd.read_csv("D:\Flipkart sales analytics\cleaned_dataset\Products.csv")
categories = pd.read_csv("D:\Flipkart sales analytics\cleaned_dataset\Categories.csv")
payments = pd.read_csv("D:\Flipkart sales analytics\cleaned_dataset\Payments.csv")
reviews = pd.read_csv("D:\Flipkart sales analytics\cleaned_dataset\Reviews.csv")
returns = pd.read_csv("D:\Flipkart sales analytics\cleaned_dataset\Returns.csv")
shipments = pd.read_csv("D:\Flipkart sales analytics\cleaned_dataset\Shipments.csv")
sellers = pd.read_csv("D:\Flipkart sales analytics\cleaned_dataset\Sellers.csv")

In [3]:
## convert date_columns data types into datetime

date_columns = [
    "order_date",
    "delivery_date",
    "return_date",
    "review_date",
    "dispatch_date"
]

for col in date_columns:

    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col])

    if col in reviews.columns:
        reviews[col] = pd.to_datetime(reviews[col])

    if col in returns.columns:
        returns[col] = pd.to_datetime(returns[col])

    if col in shipments.columns:
        shipments[col] = pd.to_datetime(shipments[col])

In [4]:
## customer features

## Age Group
# Business Meaning : Customer age is easier to analyze when grouped into business-friendly segments instead of individual ages.

bins = [18,25,35,45,55,65,100]

labels = [
    "18-25",
    "26-35",
    "36-45",
    "46-55",
    "56-65",
    "65+"
]

customers["Age_Group"] = pd.cut(
    customers["customer_age"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [5]:
customers[["customer_age", "Age_Group"]].head()

,customer_age,Age_Group
0,29,26-35
1,37,36-45
2,33,26-35
3,31,26-35
4,60,56-65


In [6]:
# Customer Full Address
# Business Meaning : Creates a complete location field that can be used in dashboards, maps, and customer segmentation.

customers["Full_Address"] = (
    customers["customer_city"] + ", "
    + customers["customer_state"]
)

customers[["customer_city", "customer_state", "Full_Address"]].head()

,customer_city,customer_state,Full_Address
0,Lucknow,Up,"Lucknow, Up"
1,Coimbatore,Tn,"Coimbatore, Tn"
2,Kolkata,Wb,"Kolkata, Wb"
3,Kolkata,Wb,"Kolkata, Wb"
4,Coimbatore,Tn,"Coimbatore, Tn"


In [7]:
# Customer Tenure (Days)
# Business Meaning : Shows how long each customer has been associated with the platform, useful for loyalty and retention analysis.

reference_date = orders["order_date"].max()

customer_first_order = (
    orders.groupby("customer_id")["order_date"]
    .min()
)

customers = customers.merge(
    customer_first_order.rename("first_order"),
    on="customer_id",
    how="left"
)

customers["Customer_Tenure_Days"] = (
    reference_date - customers["first_order"]
).dt.days

customers[["customer_id", "Customer_Tenure_Days"]] \
    .sort_values(by="Customer_Tenure_Days", ascending=False) \
    .head()

,customer_id,Customer_Tenure_Days
24999,CUST025000,1199.0
10277,CUST010278,1199.0
16239,CUST016240,1199.0
15873,CUST015874,1199.0
15712,CUST015713,1199.0


In [7]:
## Order Features

# Order Month
# Business Meaning : Supports monthly trend analysis and seasonality.

orders["Order_Month"] = orders["order_date"].dt.month_name()

# Order Quarter
# Business Meaning : Useful for quarterly business reporting

orders["Order_Quarter"] = (
    "Q" +
    orders["order_date"]
    .dt.quarter.astype(str)
)

# Order Year
orders["Order_Year"] = (
    orders["order_date"]
    .dt.year
)

# Day of Week
# Business Meaning : Helps identify whether customers purchase more during weekdays or weekends.

orders["Day_Name"] = (
    orders["order_date"]
    .dt.day_name()
)

# Weekend Order Flag
orders["Weekend_Order"] = (
    orders["Day_Name"]
    .isin(["Saturday","Sunday"])
)

In [8]:
orders.head()

,order_id,customer_id,order_date,order_status,channel,order_priority,coupon_used,coupon_discount_pct,delivery_days_actual,delivery_days_estimated,Order_Month,Order_Quarter,Order_Year,Day_Name,Weekend_Order
0,ORD0000001,CUST001996,2022-03-18 15:15:14,Delivered,Website,Low,0,0.00,4,4,March,Q1,2022,Friday,False
1,ORD0000002,CUST010311,2021-12-19 00:19:11,Delivered,App,Low,1,30.94,1,1,December,Q4,2021,Sunday,True
2,ORD0000003,CUST020207,2021-12-30 01:01:47,Delivered,Website,Low,0,0.00,4,2,December,Q4,2021,Thursday,False
3,ORD0000004,CUST016230,2021-09-03 06:15:55,Delivered,App,High,0,0.00,11,11,September,Q3,2021,Friday,False
4,ORD0000005,CUST013548,2021-12-29 05:21:56,Delivered,Website,Medium,0,0.00,2,1,December,Q4,2021,Wednesday,False


In [14]:
# ## Revenue features

# Order Value
# Business Meaning : Calculates total revenue generated by each order.

order_value = (
    order_items
    .groupby("order_id")["line_revenue"]
    .sum()
    .reset_index(name="Order_Value")
)

orders = orders.merge(
    order_value,
    on="order_id",
    how="left"
)

# Total Quantity
order_quantity = (
    order_items
    .groupby("order_id")["quantity"]
    .sum()
    .reset_index(name="Total_Items")
)

orders = orders.merge(
    order_quantity,
    on="order_id",
    how="left"
)



In [9]:
orders.head()

,order_id,customer_id,order_date,order_status,channel,order_priority,coupon_used,coupon_discount_pct,delivery_days_actual,delivery_days_estimated,Order_Month,Order_Quarter,Order_Year,Day_Name,Weekend_Order
0,ORD0000001,CUST001996,2022-03-18 15:15:14,Delivered,Website,Low,0,0.00,4,4,March,Q1,2022,Friday,False
1,ORD0000002,CUST010311,2021-12-19 00:19:11,Delivered,App,Low,1,30.94,1,1,December,Q4,2021,Sunday,True
2,ORD0000003,CUST020207,2021-12-30 01:01:47,Delivered,Website,Low,0,0.00,4,2,December,Q4,2021,Thursday,False
3,ORD0000004,CUST016230,2021-09-03 06:15:55,Delivered,App,High,0,0.00,11,11,September,Q3,2021,Friday,False
4,ORD0000005,CUST013548,2021-12-29 05:21:56,Delivered,Website,Medium,0,0.00,2,1,December,Q4,2021,Wednesday,False


In [10]:
#Returned Order Flag
#Business Meaning : Creates a simple indicator for return analysis.

returned_orders = returns["order_id"].unique()

orders["Returned"] = (
    orders["order_id"]
    .isin(returned_orders)
)


In [11]:
orders.head()

,order_id,customer_id,order_date,order_status,channel,order_priority,coupon_used,coupon_discount_pct,delivery_days_actual,delivery_days_estimated,Order_Month,Order_Quarter,Order_Year,Day_Name,Weekend_Order,Returned
0,ORD0000001,CUST001996,2022-03-18 15:15:14,Delivered,Website,Low,0,0.00,4,4,March,Q1,2022,Friday,False,False
1,ORD0000002,CUST010311,2021-12-19 00:19:11,Delivered,App,Low,1,30.94,1,1,December,Q4,2021,Sunday,True,False
2,ORD0000003,CUST020207,2021-12-30 01:01:47,Delivered,Website,Low,0,0.00,4,2,December,Q4,2021,Thursday,False,True
3,ORD0000004,CUST016230,2021-09-03 06:15:55,Delivered,App,High,0,0.00,11,11,September,Q3,2021,Friday,False,False
4,ORD0000005,CUST013548,2021-12-29 05:21:56,Delivered,Website,Medium,0,0.00,2,1,December,Q4,2021,Wednesday,False,False


In [12]:
## Export Engineered data

customers.to_csv(
    "D:\Flipkart sales analytics\cleaned_dataset\Engineered_Customers.csv",
    index=False
)

orders.to_csv(
    "D:\Flipkart sales analytics\cleaned_dataset\Engineered_Orders.csv",
    index=False
)

In [13]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 16 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   order_id                 120000 non-null  object        
 1   customer_id              120000 non-null  object        
 2   order_date               120000 non-null  datetime64[ns]
 3   order_status             120000 non-null  object        
 4   channel                  120000 non-null  object        
 5   order_priority           120000 non-null  object        
 6   coupon_used              120000 non-null  int64         
 7   coupon_discount_pct      120000 non-null  float64       
 8   delivery_days_actual     120000 non-null  int64         
 9   delivery_days_estimated  120000 non-null  int64         
 10  Order_Month              120000 non-null  object        
 11  Order_Quarter            120000 non-null  object        
 12  Order_Year      

In [14]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   customer_id        25000 non-null  object  
 1   customer_gender    25000 non-null  object  
 2   customer_age       25000 non-null  int64   
 3   customer_state     25000 non-null  object  
 4   customer_city      25000 non-null  object  
 5   customer_pincode   25000 non-null  int64   
 6   signup_date        25000 non-null  object  
 7   customer_segment   25000 non-null  object  
 8   preferred_device   25000 non-null  object  
 9   preferred_payment  25000 non-null  object  
 10  Age_Group          25000 non-null  category
 11  Full_Address       25000 non-null  object  
dtypes: category(1), int64(2), object(9)
memory usage: 2.1+ MB
